# G06: Sparse Autoencoders — Discovering Features

**Prerequisites:** G02 (loading GPT-2, cache), G05 (MLPs and the polysemanticity problem).

**What you'll learn:**
- The superposition hypothesis: why models represent more features than they have dimensions
- Evidence for superposition in GPT-2 activations
- How sparse autoencoders (SAEs) decompose activations into interpretable features
- Training an SAE on GPT-2 layer activations
- Analyzing discovered features: what do they respond to?
- Feature→logit effects: how do features influence predictions?
- The reconstruction quality vs. sparsity tradeoff

G05 showed us that individual neurons are polysemantic — they respond to multiple unrelated inputs. This tutorial introduces sparse autoencoders, the leading approach to finding the "true" features that models use internally.

---

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import vis
from irtk.superposition import dimensionality_analysis
from irtk.sae import SparseAutoencoder, train_sae, feature_activation_stats, top_activating_examples, feature_logit_attribution

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"GPT-2 Small: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")

## The Superposition Hypothesis

Neural networks represent **more features than they have dimensions**. GPT-2 Small has 768-dimensional residual stream vectors, but it needs to track thousands of concepts — syntax, semantics, facts, reasoning patterns. How does it fit all of that into 768 numbers?

The answer is **superposition**: features are encoded as *directions* in activation space, not as individual neurons. Multiple features share the same neurons by pointing in different (but not orthogonal) directions. This is why neurons are polysemantic — a single neuron activates when *any* of its overlapping features are present. The model can get away with this because features are **sparse**: for any given input, most features are off, so their overlapping directions rarely interfere. The key insight is that the right unit of analysis is a **direction**, not a neuron.

## Evidence from GPT-2

Let's look for evidence of superposition by analyzing the dimensionality of GPT-2's representations. If representations were random, we'd expect the effective dimensionality to equal `d_model`. If the model uses structured, lower-dimensional representations (i.e., features packed into directions), we'll see effective dimensionality *below* `d_model` — but the number of meaningful features likely *exceeds* `d_model`.

In [ ]:
# Diverse prompts to probe the model's representation structure
prompts = [
    "The quick brown fox jumps over the lazy dog",
    "In 1969, Neil Armstrong walked on the moon",
    "The capital of France is Paris",
    "Machine learning models use gradient descent",
    "Shakespeare wrote Romeo and Juliet",
    "Water freezes at zero degrees Celsius",
    "The largest ocean on Earth is the Pacific",
    "Python is a popular programming language",
    "The speed of light is very fast",
    "Dogs are loyal companions to humans",
    "The stock market crashed in October 1929",
    "Photosynthesis converts sunlight into energy",
    "Mozart composed his first symphony at age eight",
    "The human brain contains billions of neurons",
    "Democracy originated in ancient Greece",
]
token_sequences = [model.to_tokens(p) for p in prompts]

# Analyze effective dimensionality across layers
dim_results = dimensionality_analysis(model, token_sequences)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot participation ratio (effective dimensionality) per layer
axes[0].plot(dim_results["participation_ratio"], "o-", color="steelblue")
axes[0].axhline(y=model.cfg.d_model, color="red", linestyle="--", alpha=0.5, label=f"d_model={model.cfg.d_model}")
axes[0].set_xticks(range(len(dim_results["labels"])))
axes[0].set_xticklabels(dim_results["labels"], rotation=45)
axes[0].set_ylabel("Participation Ratio")
axes[0].set_title("Effective Dimensionality per Layer")
axes[0].legend()

# Plot eigenvalue spectra for a few layers
for i, label in enumerate(dim_results["labels"]):
    if label in ["embed", "L3", "L6", "L11"]:
        spectrum = dim_results["eigenvalue_spectra"][i]
        spectrum_normed = spectrum / spectrum.sum()
        axes[1].plot(spectrum_normed[:100], label=label)

axes[1].set_xlabel("Eigenvalue Index")
axes[1].set_ylabel("Fraction of Variance")
axes[1].set_title("Eigenvalue Spectra (top 100)")
axes[1].legend()
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

print(f"\nd_model = {model.cfg.d_model}")
print(f"Effective dimensionality ranges from {dim_results['participation_ratio'].min():.0f} to {dim_results['participation_ratio'].max():.0f}")
print("The model uses structured, lower-dimensional subspaces — but encodes many features via superposition.")

## Enter the Sparse Autoencoder

If features are directions (not neurons), how do we find them? A **sparse autoencoder** (SAE) learns to decompose activation vectors into interpretable feature directions.

The architecture is simple:
1. **Encoder**: project a residual stream vector (`d_model` dims) UP to a much larger space (`n_features >> d_model`) through a linear layer + ReLU. This produces **sparse feature activations** — most features are zero.
2. **Decoder**: project back DOWN from the feature space to reconstruct the original activation vector.

The training loss has two terms:
- **Reconstruction error** (MSE): the SAE should faithfully reconstruct the original activations.
- **L1 penalty** on feature activations: encourages sparsity — only a few features should fire for any input.

Each column of the decoder weight matrix **is** a feature direction in the original `d_model`-dimensional space. The L1 penalty is what makes these features interpretable: without it, the autoencoder would learn entangled, uninterpretable features.

## Collecting Training Data

To train an SAE, we need many activation vectors from the model. We'll run GPT-2 on diverse prompts and collect the residual stream at **layer 6** — deep enough to have rich, abstract features, but not so deep that representations have become highly specialized for next-token prediction.

In [ ]:
# Diverse prompts to collect activations from
texts = [
    "The capital of France is Paris and it is beautiful",
    "In mathematics, two plus two equals four always",
    "The president of the United States lives in the White House",
    "Dogs and cats are popular pets around the world",
    "Python is a programming language used for machine learning",
    "The sun rises in the east and sets in the west",
    "Shakespeare wrote many famous plays including Hamlet and Macbeth",
    "The speed of light is approximately three hundred thousand km",
    "Water boils at one hundred degrees Celsius at sea level",
    "The largest planet in our solar system is Jupiter which is huge",
    "Einstein developed the theory of relativity in the early twentieth century",
    "Music has been part of human culture for thousands of years",
    "The Amazon rainforest is the largest tropical forest on Earth",
    "Computers process information using binary ones and zeros internally",
    "The Great Wall of China is visible from space according to myth",
    "Gravity keeps the planets in orbit around the sun continuously",
]

# Collect layer 6 residual stream activations
target_layer = 6
all_activations = []

for text in texts:
    tokens = model.to_tokens(text)
    _, cache = model.run_with_cache(tokens)
    # Get residual stream after layer 6
    resid = np.array(cache[("resid_post", target_layer)])  # [seq_len, d_model]
    all_activations.append(resid)

activations = np.concatenate(all_activations, axis=0)
print(f"Collected {activations.shape[0]} activation vectors of dim {activations.shape[1]}")
activations_jax = jnp.array(activations)

## Training the SAE

The key hyperparameters:
- **`n_features`**: how many features the SAE can learn. We use 4× overcomplete (`n_features = 4 * d_model`), meaning 3072 features for 768 dimensions. Real research uses 10–100× overcomplete.
- **`l1_coeff`**: the sparsity pressure. Higher values → fewer active features per input (more interpretable), but worse reconstruction.
- **`lr`** and **`epochs`**: standard training knobs.

The tradeoff is fundamental: stronger L1 → sparser, more interpretable features, but the SAE can't use enough features to reconstruct perfectly.

In [ ]:
n_features = model.cfg.d_model * 4  # 4x overcomplete: 3072 features

result = train_sae(
    activations_jax,
    d_model=model.cfg.d_model,
    n_features=n_features,
    l1_coeff=3e-3,
    lr=3e-4,
    epochs=50,
    batch_size=64,
    verbose=True,
)
sae = result.sae
print(f"\nTrained SAE: {model.cfg.d_model} -> {n_features} features")
print(f"Final reconstruction loss: {result.recon_losses[-1]:.6f}")
print(f"Final L1 loss: {result.l1_losses[-1]:.6f}")

In [ ]:
# Plot training curves
vis.plot_sae_training(
    result.train_losses,
    result.recon_losses,
    result.l1_losses,
    result.l0_sparsities,
    title="SAE Training on Layer 6 Activations",
)
plt.show()

## Analyzing Features

Now the exciting part: what did the SAE learn? Let's look at the statistics of the discovered features — how often they fire, how many are dead, and how sparse the representations are.

In [ ]:
# Get per-feature statistics
stats = feature_activation_stats(sae, activations_jax)

firing_rate = stats["firing_rate"]
mean_acts = stats["mean_acts"]
max_acts = stats["max_acts"]

print(f"Average features active per input (L0): {stats['l0_mean']:.1f} / {n_features}")
print(f"Dead features (never fire): {(firing_rate == 0).sum()} / {n_features}")
print(f"Rare features (fire < 5% of time): {(firing_rate < 0.05).sum()} / {n_features}")
print(f"Common features (fire > 50% of time): {(firing_rate > 0.5).sum()} / {n_features}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Firing rate distribution
axes[0].hist(firing_rate[firing_rate > 0], bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Firing Rate")
axes[0].set_ylabel("Number of Features")
axes[0].set_title("Feature Firing Rate Distribution")

# Mean activation distribution (for active features)
axes[1].hist(mean_acts[mean_acts > 0], bins=50, color="coral", edgecolor="white")
axes[1].set_xlabel("Mean Activation")
axes[1].set_ylabel("Number of Features")
axes[1].set_title("Mean Activation (Active Features)")

# Max activation distribution
axes[2].hist(max_acts[max_acts > 0], bins=50, color="mediumseagreen", edgecolor="white")
axes[2].set_xlabel("Max Activation")
axes[2].set_ylabel("Number of Features")
axes[2].set_title("Max Activation (Active Features)")

plt.tight_layout()
plt.show()

In [ ]:
# Examine individual features: what tokens/contexts activate them?
# Pick the top most-frequently-active features
active_feature_indices = np.argsort(firing_rate)[::-1]

# We need a mapping from activation index back to (text_index, position)
activation_map = []  # (text_idx, position, token_str)
for text_idx, text in enumerate(texts):
    tokens = model.to_tokens(text)
    for pos in range(tokens.shape[0]):
        tok_str = tokenizer.decode([int(tokens[pos])])
        activation_map.append((text_idx, pos, tok_str))

print("Top 5 most frequently active features:\n")
for rank in range(5):
    feat_idx = int(active_feature_indices[rank])
    top_indices, top_values = top_activating_examples(sae, feat_idx, activations_jax, k=8)
    
    print(f"Feature {feat_idx} (fires {firing_rate[feat_idx]*100:.1f}% of the time):")
    contexts = []
    for idx, val in zip(top_indices, top_values):
        text_idx, pos, tok_str = activation_map[int(idx)]
        contexts.append(f"  '{tok_str.strip()}' (act={val:.3f}) in: ...{texts[text_idx][max(0,pos*4-20):pos*4+20]}...")
    for c in contexts[:6]:
        print(c)
    print()

## Feature→Logit Effects

Each SAE feature has a **decoder direction** — a vector in `d_model` space. We can project this direction through the unembedding matrix (`W_U`) to see which vocabulary tokens this feature promotes or suppresses.

This is the SAE analog of projecting a neuron direction through the unembedding. If a feature is truly monosemantic (responds to one concept), we'd expect its top promoted tokens to be semantically coherent — all geography tokens, or all number tokens, or all code-related tokens.

In [ ]:
# Feature→Logit analysis: what tokens does each feature promote/suppress?
W_U = model.unembed.W_U  # [d_model, d_vocab]

# Analyze the top 5 most active features
print("Feature → Logit Analysis")
print("=" * 60)

for rank in range(5):
    feat_idx = int(active_feature_indices[rank])
    promoted, suppressed = feature_logit_attribution(sae, W_U, feat_idx, k=10)
    
    promoted_tokens = [tokenizer.decode([int(t)]) for t in promoted]
    suppressed_tokens = [tokenizer.decode([int(t)]) for t in suppressed]
    
    print(f"\nFeature {feat_idx} (fires {firing_rate[feat_idx]*100:.1f}%):")
    print(f"  Promotes: {promoted_tokens}")
    print(f"  Suppresses: {suppressed_tokens}")

## Reconstruction Quality

There's a fundamental tradeoff at the heart of SAE training: **more sparsity means more interpretable features, but worse reconstruction**. The L1 penalty prevents the SAE from using too many features simultaneously, which means some information is lost.

Let's measure how well the SAE reconstructs the original activations, and check whether the model's predictions change when we substitute reconstructed activations for the originals.

In [ ]:
# Reconstruction quality analysis
reconstructed, feature_acts = sae(activations_jax)

# Per-sample MSE
mse_per_sample = jnp.mean((activations_jax - reconstructed) ** 2, axis=-1)

# Cosine similarity between original and reconstructed
cos_sim = jnp.sum(activations_jax * reconstructed, axis=-1) / (
    jnp.linalg.norm(activations_jax, axis=-1) * jnp.linalg.norm(reconstructed, axis=-1) + 1e-8
)

# Fraction of variance explained
total_var = jnp.var(activations_jax)
residual_var = jnp.var(activations_jax - reconstructed)
frac_variance_explained = 1.0 - residual_var / total_var

print(f"Reconstruction Quality:")
print(f"  Mean MSE: {float(jnp.mean(mse_per_sample)):.6f}")
print(f"  Mean cosine similarity: {float(jnp.mean(cos_sim)):.4f}")
print(f"  Fraction of variance explained: {float(frac_variance_explained):.4f}")
print(f"  Mean L0 (active features): {float(jnp.mean(jnp.sum(feature_acts > 0, axis=-1))):.1f} / {n_features}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(np.array(cos_sim), bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Cosine Similarity")
axes[0].set_ylabel("Count")
axes[0].set_title("Original vs Reconstructed (Cosine Sim)")
axes[0].axvline(x=float(jnp.mean(cos_sim)), color="red", linestyle="--", label=f"mean={float(jnp.mean(cos_sim)):.3f}")
axes[0].legend()

axes[1].hist(np.array(mse_per_sample), bins=50, color="coral", edgecolor="white")
axes[1].set_xlabel("MSE")
axes[1].set_ylabel("Count")
axes[1].set_title("Per-Sample Reconstruction Error")
axes[1].axvline(x=float(jnp.mean(mse_per_sample)), color="red", linestyle="--", label=f"mean={float(jnp.mean(mse_per_sample)):.4f}")
axes[1].legend()

plt.tight_layout()
plt.show()

## Scaling and Limitations

Be honest about what we've done here and what real SAE research looks like:

- **Our SAE is tiny.** We trained on ~200 activation vectors from 16 prompts with a 4× overcomplete factor. Real SAE research (Bricken et al. 2023, Templeton et al. 2024) uses millions of tokens and 10–100× overcomplete factors.
- **More data and features improve interpretability.** With more training data, SAE features become more monosemantic — individual features respond to single, coherent concepts rather than mixtures.
- **The core insight holds regardless of scale.** Features exist as directions in activation space, superposition is real, and SAEs can recover interpretable features. Our small-scale demo shows the mechanics; scaling up delivers the quality.
- **Active research area.** Training dynamics (dead features, feature splitting), evaluation of feature interpretability, and how to use SAE features for downstream tasks are all open problems.

## Key Takeaways

1. **Models represent more features than dimensions** — this is the superposition hypothesis, and dimensionality analysis provides evidence for it.
2. **SAEs decompose activations into sparse, interpretable feature directions** by training an overcomplete autoencoder with an L1 sparsity penalty.
3. **Feature→logit analysis reveals each feature's effect on predictions** by projecting decoder directions through the unembedding.
4. **Sparsity and reconstruction quality trade off** — stronger L1 pressure means more interpretable but less faithful features.
5. **SAE features are often more interpretable than individual neurons**, because they recover the model's "true" computational units rather than arbitrary basis directions.

| Function | Purpose |
|---|---|
| `dimensionality_analysis` | Measure effective dimensionality of representations |
| `train_sae` | Train a sparse autoencoder on activation vectors |
| `feature_activation_stats` | Get firing rate, mean/max activation per feature |
| `top_activating_examples` | Find which inputs most activate a given feature |
| `feature_logit_attribution` | Project feature directions through the unembedding |
| `vis.plot_sae_training` | Visualize SAE training curves |

## Exercises

1. **Different layers:** Train SAEs on layers 0, 3, 9, and 11. Do earlier layers have simpler, more token-level features while later layers have more abstract features?
2. **Overcomplete factor:** Try 2× and 8× overcomplete. How does the number of dead features change? Does 8× produce more monosemantic features?
3. **Sparsity pressure:** Vary `l1_coeff` from 1e-4 to 1e-2. Plot reconstruction quality vs. L0 sparsity. Where is the sweet spot?
4. **Feature testing:** Find a feature that seems to respond to a specific concept (e.g., geography, numbers, code). Write new test prompts and verify that the feature fires when expected and stays silent otherwise.

## Further Reading

- Bricken et al. 2023, [Towards Monosemanticity](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — the landmark paper on training SAEs on language model activations.
- Cunningham et al. 2023, [Sparse Autoencoders Find Highly Interpretable Features](https://arxiv.org/abs/2309.08600) — systematic study of SAE feature interpretability across scales.
- Templeton et al. 2024, [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/) — SAEs scaled to Claude, finding millions of interpretable features.

**Next:** [G07](G07_circuits.ipynb) — using the features and tools we've built up to trace full circuits through the model.